# CoPTrS Denoising Autoencoder Pipeline

To identify coordinated pairs of transcription start sites (CoPTrS) across the human promoterome, we developed CoPTrS — a denoising autoencoder pipeline. This
notebook documents the architecture, training procedure, deployment of the pipeline and *k*-means clustering.

CAGE scaled scores and associated derivatives used as input features were curated by Satish et al. (2025, unpublished; DOI: [10.64898/2025.12.22.696121](https://doi.org/10.64898/2025.12.22.696121)) and are not distributed within this repository.

> **Note:** Pre-trained autoencoder weights and derived latent vectors are not included in this repository. Access will be provided upon reasonable request to the corresponding author.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import time
import random
from concurrent.futures import ProcessPoolExecutor, as_completed

# ── Data handling ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import polars as pl

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── Utilities ─────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from tqdm import tqdm

## TSS pair covariance calculation

Covariance is calculated for all unique TSS pairs using scaled CAGE scores. Only samples where both TSSs in a pair have non-zero scaled scores are included; samples with a zero score for either TSS are excluded prior to calculation. Self-pairs and duplicate pairs are omitted.

In [ ]:
def get_covariance_edgelist(df, threshold=None):
    """
    Calculate pairwise covariance from a scaled CAGE score DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Rows = samples, columns = TSSs. Zeros are treated as non-expressed
        and excluded from covariance calculation.
    threshold : float, optional
        If provided, retain only pairs where |covariance| >= threshold.

    Returns
    -------
    pd.DataFrame
        Edge list with columns: TSS1, TSS2, Covariance (unique pairs only).
    """
    # Treat zeros as missing — covariance uses only co-expressed samples
    cov_matrix = df.replace(0, np.nan).T.cov()

    # Extract upper triangle indices (k=1 excludes diagonal / self-pairs)
    upper_indices = np.triu_indices_from(cov_matrix, k=1)

    edgelist = pd.DataFrame({
        'TSS1': cov_matrix.index.values[upper_indices[0]],
        'TSS2': cov_matrix.columns.values[upper_indices[1]],
        'Covariance': cov_matrix.values[upper_indices]
    })

    # Drop pairs with no overlapping non-zero samples
    edgelist = edgelist.dropna(subset=['Covariance'])

    if threshold is not None:
        edgelist = edgelist[edgelist['Covariance'].abs() >= threshold]

    return edgelist

In [ ]:
# Scaled CAGE score files are not distributed with this repository.
# Set paths below before running; contact authors to obtain the data.
normal_scaled_cage_scores = None   # path to Non-cancer scaled CAGE scores (.tsv)
cancer_scaled_cage_scores = None   # path to Cancer scaled CAGE scores (.tsv)

normal_df = pd.read_csv(normal_scaled_cage_scores, sep='\t', index_col=0)
cancer_df = pd.read_csv(cancer_scaled_cage_scores, sep='\t', index_col=0)

print("Processing Non-cancer cohort...")
normal_edges = get_covariance_edgelist(normal_df)

print("Processing Cancer cohort...")
cancer_edges = get_covariance_edgelist(cancer_df)

print(f"Non-cancer edges: {len(normal_edges):,}")
print(f"Cancer edges:     {len(cancer_edges):,}")

## Training data preparation

A 10% random sample of the full covariance edge list is used for model training. Within this subset, 90% is used for training and 10% for validation.

This pipeline is applied identically to both Non-cancer and Cancer covariance outputs, producing two independent models: a **Non-cancer model** and a **Cancer model**.

Feature engineering derives distance-normalised covariance ratios and co-expression indices for each TSS pair, using TSS-level annotations (*TCI, BCI, sH, gH, vd*)
established by Satish et al

In [ ]:
def flip_tss_by_distance(df):
    """
    Canonicalise TSS pair orientation so TSS1 always has the greater distance.

    This ensures each pair has a consistent orientation regardless of the order
    in which TSSs appear in the covariance edge list, preventing duplicate pairs
    with swapped TSS1/TSS2 from being treated as distinct during training.

    Parameters
    ----------
    df : pl.DataFrame
        Must contain columns: TSS1, TSS2, TSS1-Distance, TSS2-Distance.

    Returns
    -------
    pl.DataFrame
        Same schema, with TSS1/TSS2 and their distances reordered so that
        TSS1-Distance >= TSS2-Distance.
    """
    return df.with_columns([
        pl.when(pl.col("TSS1-Distance") >= pl.col("TSS2-Distance"))
            .then(pl.col("TSS1")).otherwise(pl.col("TSS2")).alias("TSS1"),
        pl.when(pl.col("TSS1-Distance") >= pl.col("TSS2-Distance"))
            .then(pl.col("TSS2")).otherwise(pl.col("TSS1")).alias("TSS2"),
        pl.when(pl.col("TSS1-Distance") >= pl.col("TSS2-Distance"))
            .then(pl.col("TSS1-Distance")).otherwise(pl.col("TSS2-Distance")).alias("TSS1-Distance"),
        pl.when(pl.col("TSS1-Distance") >= pl.col("TSS2-Distance"))
            .then(pl.col("TSS2-Distance")).otherwise(pl.col("TSS1-Distance")).alias("TSS2-Distance"),
    ])

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Covariance parquet files are not distributed with this repository.
# Set paths below; contact authors to obtain the data.
# Run this section twice: once with Non-cancer paths, once with Cancer paths.
COVARIANCE_PATH  = None   # path to cohort covariance parquet (Non-cancer or Cancer)
ANNOTATION_PATH  = None   # path to TSS_d_TCI_BCI_sH_gH_vd.tsv (Satish et al.)
TRAINING_OUT     = None   # output path for the training feature table (.tsv)

# ── Sampling ───────────────────────────────────────────────────────────────────
TRAINING_SAMPLE_N = None  # set to ~10% of total pairs for the relevant cohort
RANDOM_SEED       = 42

# Load covariance edge list and draw a 10% sample for training
covariance_full = pl.read_parquet(COVARIANCE_PATH)

random_covariance = covariance_full.sample(
    n=TRAINING_SAMPLE_N,
    with_replacement=False,
    shuffle=True,
    seed=RANDOM_SEED
)

covariance_derivative = random_covariance.select(["TSS1", "TSS2", "Covariance"])
del covariance_full, random_covariance

# Parse TSS distance from the TSS identifier (format: <gene>-<distance>)
covariance_derivative = covariance_derivative.with_columns([
    pl.col("TSS1").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS1-Distance"),
    pl.col("TSS2").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS2-Distance"),
])

# Canonicalise pair orientation: TSS1 always has the greater distance
covariance_derivative = flip_tss_by_distance(covariance_derivative)

In [ ]:
# Load TSS-level annotations (TCI, BCI, sH, gH, vd) from Satish et al.
bci = pl.read_csv(
    ANNOTATION_PATH,
    separator="\t"
).with_columns([
    pl.col("distance").cast(pl.Float64),
    (pl.col("p.gene") + "-" + pl.col("distance").cast(pl.Utf8)).alias("TSS")
]).drop(["p.gene", "distance"]).select([
    "TSS", "TCI_N", "BCI_N", "sH_N", "gH_N", "vd_N"
]).rename({
    "TCI_N": "TCI", "BCI_N": "BCI",
    "sH_N": "sH", "gH_N": "gH", "vd_N": "vd"
})

# Join annotations for TSS1 and TSS2 separately
bci_derivative = covariance_derivative.select(["TSS1", "TSS2"])

bci_derivative = bci_derivative.join(
    bci.rename({"TSS": "TSS1", "TCI": "TCI1", "BCI": "BCI1",
                "sH": "sH1", "gH": "gH1", "vd": "vd1"}),
    on="TSS1"
)
bci_derivative = bci_derivative.join(
    bci.rename({"TSS": "TSS2", "TCI": "TCI2", "BCI": "BCI2",
                "sH": "sH2", "gH": "gH2", "vd": "vd2"}),
    on="TSS2"
)

# Re-parse distances and re-canonicalise after join
bci_derivative = bci_derivative.with_columns([
    pl.col("TSS1").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS1-Distance"),
    pl.col("TSS2").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS2-Distance"),
])
bci_derivative = flip_tss_by_distance(bci_derivative)

# Merge covariance and annotation tables on canonical TSS pair
merged = covariance_derivative.join(bci_derivative, on=["TSS1", "TSS2"], how="inner")

# Compute distance ratio features (dA = TSS1-Distance >= dB = TSS2-Distance by construction)
merged = merged.with_columns([
    (pl.max_horizontal("TSS1-Distance", "TSS2-Distance") /
     pl.min_horizontal("TSS1-Distance", "TSS2-Distance")).alias("dA/dB"),
    (pl.min_horizontal("TSS1-Distance", "TSS2-Distance") /
     pl.max_horizontal("TSS1-Distance", "TSS2-Distance")).alias("dB/dA"),
])

In [ ]:
# Derive training features:
#   CdR*  — covariance normalised by distance, weighted by distance ratio
#   CT*   — covariance normalised by TCI, weighted by distance ratio
#   CB*   — covariance normalised by BCI, weighted by distance ratio
#   CsH*  — covariance normalised by sH,  weighted by distance ratio
#   CgH*  — covariance normalised by gH,  weighted by distance ratio
#   Cvd*  — covariance normalised by vd,  weighted by distance ratio
#   PCI*  — pairwise co-expression index relative to each TSS
#   CdR   — covariance normalised by log2 of the distance ratio
training_data = merged.with_columns([
    ((pl.col("Covariance") / pl.col("TSS1-Distance")) * pl.col("dA/dB")).alias("CdRA"),
    ((pl.col("Covariance") / pl.col("TSS2-Distance")) * pl.col("dB/dA")).alias("CdRB"),
    ((pl.col("Covariance") / pl.col("TCI1"))          * pl.col("dA/dB")).alias("CTRA"),
    ((pl.col("Covariance") / pl.col("TCI2"))          * pl.col("dB/dA")).alias("CTRB"),
    ((pl.col("Covariance") / pl.col("BCI1"))          * pl.col("dA/dB")).alias("CBRA"),
    ((pl.col("Covariance") / pl.col("BCI2"))          * pl.col("dB/dA")).alias("CBRB"),
    ((pl.col("Covariance") / pl.col("sH1"))           * pl.col("dA/dB")).alias("CsHRA"),
    ((pl.col("Covariance") / pl.col("sH2"))           * pl.col("dB/dA")).alias("CsHRB"),
    ((pl.col("Covariance") / pl.col("gH1"))           * pl.col("dA/dB")).alias("CgHRA"),
    ((pl.col("Covariance") / pl.col("gH2"))           * pl.col("dB/dA")).alias("CgHRB"),
    ((pl.col("Covariance") / pl.col("vd1"))           * pl.col("dA/dB")).alias("CvdRA"),
    ((pl.col("Covariance") / pl.col("vd2"))           * pl.col("dB/dA")).alias("CvdRB"),
    (((pl.col("TCI1") + pl.col("TCI2")) - (pl.col("TCI1") * pl.col("TCI2"))) /
      pl.col("TCI1")).alias("PCIA"),
    (((pl.col("TCI1") + pl.col("TCI2")) - (pl.col("TCI1") * pl.col("TCI2"))) /
      pl.col("TCI2")).alias("PCIB"),
    (pl.col("Covariance") /
     (pl.max_horizontal("TSS1-Distance", "TSS2-Distance") /
      pl.min_horizontal("TSS1-Distance", "TSS2-Distance")).log(2)).alias("CdR"),
])

# Drop intermediate columns carried over from merged
training_data = training_data.drop(
    [col for col in merged.columns if col in training_data.columns]
)

training_data.write_csv(
    TRAINING_OUT,
    separator="\t",
    include_header=True
)
print(f"Training data saved: {training_data.height:,} pairs x {len(training_data.columns)} features")

## Model training: Autoencoder

An autoencoder is trained to learn a compressed latent representation of TSS pair features. The model is trained independently on Non-cancer CAGE data and Cancer CAGE data, giving rise to two separate models: Non-cancer and Cancer.
To train each model, run this section with the corresponding training data and model name set in the configuration cell below.

**Architecture:** 15-dimensional input -> encoder (12 -> 8 -> 4) -> 4-dimensional latent
space -> decoder (4 -> 8 -> 12 -> 15). Parametric ReLU (PReLU) activations and BatchNorm are used at each layer.

**Features used:**
$$
\begin{array}{ll}
{CdR}_A & \left(\dfrac{\mathrm{Covariance}}{d_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CdR}_B & \left(\dfrac{\mathrm{Covariance}}{d_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{CTR}_A & \left(\dfrac{\mathrm{Covariance}}{TCI_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CTR}_B & \left(\dfrac{\mathrm{Covariance}}{TCI_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{CBR}_A & \left(\dfrac{\mathrm{Covariance}}{BCI_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CBR}_B & \left(\dfrac{\mathrm{Covariance}}{BCI_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{CsHR}_A & \left(\dfrac{\mathrm{Covariance}}{sH_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CsHR}_B & \left(\dfrac{\mathrm{Covariance}}{sH_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{CgHR}_A & \left(\dfrac{\mathrm{Covariance}}{gH_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CgHR}_B & \left(\dfrac{\mathrm{Covariance}}{gH_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{CvdR}_A & \left(\dfrac{\mathrm{Covariance}}{vd_A}\right)\times\dfrac{d_A}{d_B} \\[12pt]
{CvdR}_B & \left(\dfrac{\mathrm{Covariance}}{vd_B}\right)\times\dfrac{d_B}{d_A} \\[12pt]
{PCI}_A & \dfrac{(\mathrm{TCI}_A + \mathrm{TCI}_B) - (\mathrm{TCI}_A \times \mathrm{TCI}_B)}{\mathrm{TCI}_A} \\[12pt]
{PCI}_B & \dfrac{(\mathrm{TCI}_A + \mathrm{TCI}_B) - (\mathrm{TCI}_A \times \mathrm{TCI}_B)}{\mathrm{TCI}_B}
\end{array}
$$

**Training setup:**
- Loss: Mean Absolute Error (L1)
- Optimizer: Adam (lr=1e-5, weight_decay=5e-5, betas=(0.9, 0.95))
- Scheduler: Cosine Annealing
- Early stopping: patience of 5 epochs on validation loss
- Train/validation split: 90/10 (scaler fit on training set only)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Set TRAINING_DATA_PATH to the Non-cancer or Cancer training feature table
# and MODEL_NAME accordingly before running.
TRAINING_DATA_PATH = None   # path to training feature table (.tsv)
MODEL_OP_DIR       = None   # directory where model outputs will be saved
MODEL_NAME         = None   # e.g. 'CoPTR-Noncancer' or 'CoPTR-Cancer'

# ── Hyperparameters ────────────────────────────────────────────────────────────
NUM_EPOCHS   = 100
BATCH_SIZE   = 524288   # tune based on available GPU memory
PATIENCE     = 5        # early stopping patience
INPUT_DIM    = 15       # number of engineered features
LATENT_DIM   = 4        # bottleneck dimension

FINAL_PATH = os.path.join(MODEL_OP_DIR, f"Model-{MODEL_NAME}")
os.makedirs(FINAL_PATH, exist_ok=True)

In [ ]:
# Load training data; drop nulls and infinite values before splitting
training_data = pl.read_csv(TRAINING_DATA_PATH, separator="\t")
training_data = training_data.drop_nulls()
training_data = training_data.filter(
    ~pl.any_horizontal(pl.col("*").is_infinite())
)

# Shuffle then split 90/10 into train and validation
shuffled = training_data.sample(fraction=1.0, shuffle=True, seed=58)
train_idx = int(0.9 * shuffled.height)
train_df_unscaled = shuffled[:train_idx]
val_df_unscaled   = shuffled[train_idx:]

# Fit StandardScaler on training set only, then transform both sets
scaler = StandardScaler()
scaler.fit(train_df_unscaled.to_pandas())

train_df = pl.DataFrame(
    scaler.transform(train_df_unscaled.to_pandas()),
    schema=train_df_unscaled.columns
)
val_df = pl.DataFrame(
    scaler.transform(val_df_unscaled.to_pandas()),
    schema=val_df_unscaled.columns
)

print(f"Training samples:   {train_df.height:,}")
print(f"Validation samples: {val_df.height:,}")

In [ ]:
train_tensor = torch.tensor(train_df.to_numpy(), dtype=torch.float32)
val_tensor   = torch.tensor(val_df.to_numpy(),   dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=32,
    pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(val_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=32,
    pin_memory=True
)

In [ ]:
class Autoencoder(nn.Module):
    """
    Symmetric autoencoder with PReLU activations and BatchNorm.

    Encoder: input_dim -> 12 -> 8 -> latent_dim
    Decoder: latent_dim -> 8 -> 12 -> input_dim

    Parameters
    ----------
    input_dim  : int -- number of input features (15)
    latent_dim : int -- size of the bottleneck layer (4)
    """
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 12),
            nn.BatchNorm1d(12),
            nn.PReLU(),
            nn.Linear(12, 8),
            nn.BatchNorm1d(8),
            nn.PReLU(),
            nn.Linear(8, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.BatchNorm1d(8),
            nn.PReLU(),
            nn.Linear(8, 12),
            nn.BatchNorm1d(12),
            nn.PReLU(),
            nn.Linear(12, input_dim)
        )

    def forward(self, x):
        z     = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model     = Autoencoder(input_dim=INPUT_DIM, latent_dim=LATENT_DIM).to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=5e-5, betas=(0.9, 0.95))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
train_losses, val_losses = [], []
best_val_loss    = float('inf')
patience_counter = 0
best_model_path  = os.path.join(FINAL_PATH, f"{MODEL_NAME}-Best.pth")

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # -- Training --------------------------------------------------------------
    model.train()
    running_train_loss = 0.0
    for (x,) in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        x_hat, _ = model(x)
        loss = criterion(x_hat, x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_train_loss += loss.item() * x.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # -- Validation ------------------------------------------------------------
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for (x_val,) in val_loader:
            x_val = x_val.to(device)
            x_hat_val, _ = model(x_val)
            running_val_loss += criterion(x_hat_val, x_val).item() * x_val.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    scheduler.step()

    elapsed = time.time() - start
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] {elapsed:.1f}s -- "
          f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}")

    # -- Early stopping --------------------------------------------------------
    if epoch_val_loss < best_val_loss:
        best_val_loss    = epoch_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  Best model saved (val loss: {best_val_loss:.6f})")
    else:
        patience_counter += 1
        print(f"  No improvement for {patience_counter}/{PATIENCE} epoch(s)")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("Training complete.")

In [ ]:
epochs_ran = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))
plt.plot(epochs_ran, train_losses, 'o-', label='Training Loss',   color='dodgerblue')
plt.plot(epochs_ran, val_losses,   'o-', label='Validation Loss', color='tomato')
plt.title(f"Training and Validation Loss -- {MODEL_NAME}", fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('L1 Loss', fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FINAL_PATH, f"{MODEL_NAME}-Loss-Curve.png"))
plt.show()

# Save loss table
pd.DataFrame({
    'Epoch':      list(epochs_ran),
    'Train_Loss': train_losses,
    'Val_Loss':   val_losses
}).to_csv(
    os.path.join(FINAL_PATH, f"{MODEL_NAME}-Losses.tsv"),
    sep='\t', index=False
)
print("Loss curve and table saved.")

# Save full pipeline artifact: model weights + fitted scaler together.
# This single file is used for all downstream inference.
# Pipeline artifacts are not distributed; available upon request.
pipeline_artifact = {
    'model_state_dict': model.state_dict(),
    'scaler':           scaler,
    'input_dim':        INPUT_DIM,
    'latent_dim':       LATENT_DIM,
    'model_name':       MODEL_NAME,
}
torch.save(
    pipeline_artifact,
    os.path.join(FINAL_PATH, f"{MODEL_NAME}-Full-pipeline.pth")
)
print(f"Full pipeline artifact saved: {MODEL_NAME}-Full-pipeline.pth")

## Latent vector extraction (Inference)

The trained pipeline artifact (model weights + scaler, saved together as a single `.pth` file) is used to encode TSS pairs into 4-dimensional latent vectors
via the encoder only -- the decoder is not used during inference.

The full set of ~1.5 billion TSS pairs is split into two halves (Part 1 and Part 2), for the ease of handling, and latent vectors are extracted independently for each half to manage memory.
This is performed for both the Non-cancer and Cancer cohorts using their respective trained models. Both cohorts are also passed through both models, yielding the following sets of latent vectors in total:

| Pass | Input data | Model used | Output |
|------|-----------------|------------|--------|
| 1 | Non-cancer (Part 1) | Non-cancer | Latent vectors -> Non-cancer-Model + Non-cancer data, P1 |
| 2 | Non-cancer (Part 2) | Non-cancer | Latent vectors -> Non-cancer-Model + Non-cancer data, P2 |
| 3 | Cancer (Part 1) | Non-cancer | Latent vectors -> Non-cancer-Model + Cancer data, P1 |
| 4 | Cancer (Part 2) | Non-cancer | Latent vectors -> Non-cancer-Model + Cancer data, P2 |
| 5 | Cancer (Part 1) | Cancer | Latent vectors -> Cancer-Model + Cancer data, P1 |
| 6 | Cancer (Part 2) | Cancer | Latent vectors -> Cancer-Model + Cancer data, P2 |
| 7 | Non-cancer (Part 1) | Cancer | Latent vectors -> Cancer-Model + Non-cancer data, P1 |
| 8 | Non-cancer (Part 2) | Cancer | Latent vectors -> Cancer-Model + Non-cancer data, P2 |

In [ ]:
# ── Pipeline artifact (model weights + scaler) ─────────────────────────────────

PIPELINE_PATH        = None   

# ── Input: covariance data for this pass ───────────────────────────────────────

COVARIANCE_PART      = None   
ANNOTATION_PATH      = None   
PREDICTION_OUT       = None   

# ── Output: latent vectors ─────────────────────────────────────────────────────
LATENT_OUT_PATH      = None   # output path for latent vectors (.parquet)

# ── Settings ───────────────────────────────────────────────────────────────────
LABEL_COLUMN         = "TSS-Pair"
INFERENCE_BATCH_SIZE = 524288
RANDOM_SEED          = 58

np.random.seed(RANDOM_SEED)

In [ ]:
# -- Step 1: Load covariance edge list -----------------------------------------
print("Step 1: Loading covariance data...")
sampled_covariance = pl.read_parquet(COVARIANCE_PART)

covariance_derivative = sampled_covariance.with_columns([
    pl.col("TSS1").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS1-Distance"),
    pl.col("TSS2").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS2-Distance"),
])

# -- Step 2: Canonicalise pair orientation -------------------------------------
print("Step 2: Canonicalising TSS pair orientation...")
covariance_derivative = flip_tss_by_distance(covariance_derivative)

# -- Step 3: Load TSS annotations ----------------------------------------------
print("Step 3: Loading TSS annotations (TCI, BCI, gH, sH, vd)...")
bci = pl.read_csv(ANNOTATION_PATH, separator="\t").with_columns([
    pl.col("distance").cast(pl.Float64),
    (pl.col("p.gene") + "-" + pl.col("distance").cast(pl.Utf8)).alias("TSS")
]).drop(["p.gene", "distance"]).select([
    "TSS", "TCI_N", "BCI_N", "sH_N", "gH_N", "vd_N"
]).rename({
    "TCI_N": "TCI", "BCI_N": "BCI",
    "sH_N": "sH", "gH_N": "gH", "vd_N": "vd"
})

# -- Step 4: Join annotations for TSS1 and TSS2 --------------------------------
print("Step 4: Joining TSS annotations onto pairs...")
bci_derivative = covariance_derivative.select(["TSS1", "TSS2"])

bci_derivative = bci_derivative.join(
    bci.rename({"TSS": "TSS1", "TCI": "TCI1", "BCI": "BCI1",
                "sH": "sH1", "gH": "gH1", "vd": "vd1"}),
    on="TSS1"
)
bci_derivative = bci_derivative.join(
    bci.rename({"TSS": "TSS2", "TCI": "TCI2", "BCI": "BCI2",
                "sH": "sH2", "gH": "gH2", "vd": "vd2"}),
    on="TSS2"
)

bci_derivative = bci_derivative.with_columns([
    pl.col("TSS1").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS1-Distance"),
    pl.col("TSS2").str.split("-").list.get(-1).cast(pl.Float64).alias("TSS2-Distance"),
])
bci_derivative = flip_tss_by_distance(bci_derivative)

# -- Step 5: Merge and compute distance ratios ---------------------------------
print("Step 5: Merging and computing distance ratios...")
merged = covariance_derivative.join(bci_derivative, on=["TSS1", "TSS2"], how="inner")

merged = merged.with_columns([
    (pl.max_horizontal("TSS1-Distance", "TSS2-Distance") /
     pl.min_horizontal("TSS1-Distance", "TSS2-Distance")).alias("dA/dB"),
    (pl.min_horizontal("TSS1-Distance", "TSS2-Distance") /
     pl.max_horizontal("TSS1-Distance", "TSS2-Distance")).alias("dB/dA"),
])

# -- Step 6: Engineer features -------------------------------------------------
print("Step 6: Engineering features...")
prediction_data = merged.with_columns([
    ((pl.col("Covariance") / pl.col("TSS1-Distance")) * pl.col("dA/dB")).alias("CdRA"),
    ((pl.col("Covariance") / pl.col("TSS2-Distance")) * pl.col("dB/dA")).alias("CdRB"),
    ((pl.col("Covariance") / pl.col("TCI1"))          * pl.col("dA/dB")).alias("CTRA"),
    ((pl.col("Covariance") / pl.col("TCI2"))          * pl.col("dB/dA")).alias("CTRB"),
    ((pl.col("Covariance") / pl.col("BCI1"))          * pl.col("dA/dB")).alias("CBRA"),
    ((pl.col("Covariance") / pl.col("BCI2"))          * pl.col("dB/dA")).alias("CBRB"),
    ((pl.col("Covariance") / pl.col("sH1"))           * pl.col("dA/dB")).alias("CsHRA"),
    ((pl.col("Covariance") / pl.col("sH2"))           * pl.col("dB/dA")).alias("CsHRB"),
    ((pl.col("Covariance") / pl.col("gH1"))           * pl.col("dA/dB")).alias("CgHRA"),
    ((pl.col("Covariance") / pl.col("gH2"))           * pl.col("dB/dA")).alias("CgHRB"),
    ((pl.col("Covariance") / pl.col("vd1"))           * pl.col("dA/dB")).alias("CvdRA"),
    ((pl.col("Covariance") / pl.col("vd2"))           * pl.col("dB/dA")).alias("CvdRB"),
    (((pl.col("TCI1") + pl.col("TCI2")) - (pl.col("TCI1") * pl.col("TCI2"))) /
      pl.col("TCI1")).alias("PCIA"),
    (((pl.col("TCI1") + pl.col("TCI2")) - (pl.col("TCI1") * pl.col("TCI2"))) /
      pl.col("TCI2")).alias("PCIB"),
    (pl.col("Covariance") /
     (pl.max_horizontal("TSS1-Distance", "TSS2-Distance") /
      pl.min_horizontal("TSS1-Distance", "TSS2-Distance")).log(2)).alias("CdR"),
    (pl.col("TSS1") + "*" + pl.col("TSS2")).alias("TSS-Pair"),
])

prediction_data = prediction_data.drop(
    [col for col in merged.columns if col in prediction_data.columns]
)

# -- Step 7: Save prediction-ready feature table --------------------------------
print("Step 7: Saving prediction data...")
os.makedirs(os.path.dirname(PREDICTION_OUT), exist_ok=True)
prediction_data.write_csv(PREDICTION_OUT, separator="\t")
print(f"Saved: {PREDICTION_OUT}  ({prediction_data.height:,} pairs)")

In [ ]:
# -- Load saved pipeline artifact (model + scaler) -----------------------------
print("Loading pipeline artifact...")
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline = torch.load(PIPELINE_PATH, weights_only=False)

model = Autoencoder(
    input_dim=pipeline['input_dim'],
    latent_dim=pipeline['latent_dim']
).to(device)
model.load_state_dict(pipeline['model_state_dict'])
model.eval()

scaler = pipeline['scaler']
print(f"Pipeline loaded -- model: {pipeline['model_name']}, "
      f"input_dim={pipeline['input_dim']}, "
      f"latent_dim={pipeline['latent_dim']}, device={device}")

# -- Prepare features ----------------------------------------------------------
expected_features = scaler.feature_names_in_
prediction_data   = prediction_data.filter(
    pl.all_horizontal(pl.col(expected_features).is_finite())
)
print(f"Clean pairs: {prediction_data.height:,}")

labels            = prediction_data.select(LABEL_COLUMN).to_series()
features_unscaled = prediction_data.select(expected_features).to_pandas()

assert len(features_unscaled) == len(labels), "Label/feature length mismatch after cleaning."

features_scaled = scaler.transform(features_unscaled)
print(f"Features scaled: {features_scaled.shape}")

In [ ]:
# -- Build DataLoader for batched inference ------------------------------------
inference_loader = DataLoader(
    TensorDataset(torch.tensor(features_scaled, dtype=torch.float32)),
    batch_size=INFERENCE_BATCH_SIZE,
    shuffle=False
)

# -- Pass each batch through the encoder only ----------------------------------
all_latents = []
with torch.no_grad():
    for (x,) in tqdm(inference_loader, desc="Extracting latent vectors"):
        z = model.encoder(x.to(device))
        all_latents.append(z.cpu().numpy())

latent_array = np.concatenate(all_latents, axis=0)
print(f"Latent vectors: {latent_array.shape[0]:,} pairs x {latent_array.shape[1]} dims")

# -- Assemble output DataFrame and save ----------------------------------------
latent_df = pl.DataFrame(
    latent_array,
    schema=[f"L{i+1}" for i in range(latent_array.shape[1])]
).with_columns(
    pl.Series(LABEL_COLUMN, labels)
).select([LABEL_COLUMN] + [f"L{i+1}" for i in range(latent_array.shape[1])])

os.makedirs(os.path.dirname(LATENT_OUT_PATH), exist_ok=True)
latent_df.write_parquet(LATENT_OUT_PATH)
print(f"Saved: {LATENT_OUT_PATH}")

## *k*-Means clustering on latent vectors

Latent vectors from both halves (Part 1 and Part 2) for each combination are merged into a single edge list covering all ~1.5 billion TSS pairs, and K-Means clustering is applied for *k* = 3 to 20.

In [ ]:
# -- Paths ---------------------------------------------------------------------

LATENT_PART_1  = None 
LATENT_PART_2  = None 
CLUSTERING_OUT = None 

# -- Parameters ----------------------------------------------------------------
K_MIN       = 3
K_MAX       = 20
LATENT_COLS = [f"L{i+1}" for i in range(4)]
RANDOM_SEED = 42

# -- Load and merge latent vectors ---------------------------------------------
print("Loading and merging latent vectors...")
latent_df = pl.concat([
    pl.read_parquet(LATENT_PART_1),
    pl.read_parquet(LATENT_PART_2),
], how="vertical")
print(f"Total pairs: {latent_df.height:,}")

latent_vectors  = latent_df.select(LATENT_COLS).to_numpy()
tss_pair_labels = latent_df.select("TSS-Pair").to_series()

# -- Run K-Means for k = K_MIN to K_MAX ----------------------------------------
for k in range(K_MIN, K_MAX + 1):
    print(f"Running K-Means (k={k})...")

    kmeans         = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto")
    cluster_labels = kmeans.fit_predict(latent_vectors)

    results_df = pl.DataFrame({
        "TSS-Pair":         tss_pair_labels,
        "Assigned_Cluster": cluster_labels,
    }).sort("Assigned_Cluster")

    cluster_counts = results_df["Assigned_Cluster"].value_counts().sort("Assigned_Cluster")

    out_dir = os.path.join(CLUSTERING_OUT, f"{k}-Clusters")
    os.makedirs(out_dir, exist_ok=True)

    results_df.write_parquet(    os.path.join(out_dir, f"{k}-Clusters-detailed.parquet"))
    cluster_counts.write_parquet(os.path.join(out_dir, f"{k}-Clusters-counts.parquet"))
    print(f"  k={k} saved to: {out_dir}")

print("Clustering complete for all k.")